# Tradable Long-Short Factors

A firm characteristic is not yet an investment. To trade on it, one needs to specify which portfolios to buy, which to sell, and how much exposure to allocate to each side. A tradable factor turns the comparison into a defined payoff. In practice, tradable factors have two roles. First, they represent investments whose payoffs can be studied, and second, they provide benchmarks for evaluating other investments. A positive factor return means the specified position earned a positive payoff during that period. Whether it adds abnormal expected return relative to existing opportunities requires a separate benchmark comparison, which I turn to in Notebook 05.

In this notebook, I first explain the market factor, long–short payoffs and how characteristic portfolios are combined. I then apply this framework to the Fama–French construction and examine the locally constructed factor returns. The measurement and portfolio-formation choices established in [01](01_firm_characteristics.ipynb) and [02](02_portfolio_formation.ipynb) remain the foundation of those investments.


In [ ]:
import sys
from pathlib import Path
ROOT=Path.cwd().resolve()
ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
NAVY, BLUE, GREEN, ORANGE, GRAY = '#17324D', '#2878A5', '#168A86', '#D99A32', '#737D87'
STYLE={'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':12,'axes.labelsize':11,
       'xtick.labelsize':10,'ytick.labelsize':10,'legend.fontsize':10,'figure.dpi':110,
       'figure.facecolor':'white','axes.facecolor':'white','axes.spines.top':False,
       'axes.spines.right':False,'axes.prop_cycle':plt.cycler(color=[BLUE,GREEN,ORANGE,NAVY]),
       'grid.alpha':.15,'lines.linewidth':1.8}
plt.rcParams.update(STYLE)
SIGNED=LinearSegmentedColormap.from_list('ocean_white_amber',[ORANGE,'white',BLUE])
from factor_asset_pricing.wrds_data import load_prepared_data
RAW_WRDS = ROOT / "data/raw/wrds"
PROCESSED = ROOT / "data/processed"


## 1. Tradable Factors

### 1.1 Factors as investments and benchmarks

A traded factor tells us which assets or portfolios to hold. It is different from a firm characteristic or an estimated exposure. Book-to-market is a characteristic. Buying high-book-to-market portfolios and selling low-book-to-market portfolios produces a payoff. A beta measures how another investment’s returns co-move with that payoff.

The same factor can therefore be an investment and a regressor in an asset-pricing model. Its return series describes what the specified positions earned, while a regression asks how much of another investment’s return is associated with those positions. The investment payoff and its economic explanation are separate questions. Benchmark regressions in 05 assess whether the factor earns abnormal return.

### 1.2 The market factor

The market factor is the natural starting point because it is not only central to CAPM but also well known for being difficult to beat in practice. To construct a market benchmark, we need to choose which securities to include and how to weight them. A capitalization-weighted equity portfolio gives each eligible security its share of beginning-period market value,

$$w^M_{i,t-1}=\frac{ME_{i,t-1}}{\sum_jME_{j,t-1}},\qquad
R_{M,t}=\sum_iw^M_{i,t-1}R_{i,t}.$$

$ME_{i,t-1}$ denotes pre-return market equity, and $R_{i,t}$ is the security’s simple total return. The unit-sum weights make this a fully invested risky portfolio. Subtract its matched risk-free return to obtain the market factor,

$$MKT_t=R_{M,t}-R_{f,t}.$$

We use a risk-free return with the same dates, horizon and units as the market return, and subtract it once. A US study can choose a broad US equity universe, while a country-specific study can choose its own eligible exchanges. Neither equity proxy necessarily contains all risky wealth in the theoretical CAPM market.

### 1.3 Long–short factors

A long–short factor compares two investments. We buy one portfolio and sell another short. The short position gains when the securities sold short fall in value and loses when they rise. Buying one unit of the long portfolio and shorting one unit of the short portfolio gives the payoff

$$F_t=R_{L,t}-R_{S,t}.$$

Here $R_{L,t}$ and $R_{S,t}$ are the total returns earned by the two underlying portfolios. If the long portfolio returns 3% and the shorted portfolio returns 1%, the factor earns 2%. Both portfolios increased in value, but the long side outperformed the short side. A positive factor payoff therefore requires relative outperformance, rather than a fall in the shorted securities.

#### Weighting within each leg

Weighting within each leg of a long–short factor is an important design choice. Two common approaches are equal weighting and value weighting:

$$w^{EW}_{i,P,t-1}=\frac{1}{N_{P,t}},\qquad
w^{VW}_{i,P,t-1}=\frac{ME_{i,t-1}}{\sum_{j\in\mathcal P_t}ME_{j,t-1}}.$$

Here, $P$ denotes either leg, $N_{P,t}$ is its number of stocks, and $\mathcal P_t$ is the set of stocks in that leg. As with the market factor, value weighting is the most popular approach, but the distinction is important. Consider, for example, the size factor, which goes long on small firms and short on big firms. With value weighting, the investor gives more weight to the largest firms within the small-firm portfolio and to the largest firms within the big-firm portfolio. With equal weighting, all firms within each portfolio receive the same weight.

#### Allocation between the legs

$$F_t=aR_{L,t}-bR_{S,t}.$$

Here $a$ is the allocation to the long leg and $b$ is the magnitude of the short allocation. In factor construction, we usually use $a=b=1$, giving the familiar payoff $F_t=R_{L,t}-R_{S,t}$. Equal allocations make the factor self-financing and focus the comparison on the relative performance of the two portfolios. They do not necessarily eliminate market exposure.



## 2. Example: The Fama–French Factors

For this example, I apply the construction to Fama–French portfolios formed on size, book-to-market, operating profitability and investment, a construction widely taught in business schools around the world.

### 2.1 Construction conventions

I use the prepared stock and accounting data from 00, following the measurements and formation rules in 01–02. Each June, eligible NYSE stocks determine the size median and the 30th and 70th characteristic percentiles. The resulting size × characteristic portfolios are held from July through June, with value weights based on the preceding month's market equity. The market factor uses the eligible linked stock universe and subtracts its matched risk-free return.

### 2.2 Defining the factor positions

The table summarizes the positions and how their portfolio returns enter each factor. $S$ and $B$ denote small and big firms. The remaining letters identify the characteristic group: high/low BM ($H/L$), robust/weak profitability ($R/W$), and conservative/aggressive investment ($C/A$). Stocks within each portfolio remain value weighted.

| Factor | Long position | Short position | Equation |
|---|---|---|---|
| MKT | Capitalization-weighted market | Risk-free financing | $MKT_t=R_{M,t}-R_{f,t}$ |
| HML | Half small-high BM, half big-high BM | Half small-low BM, half big-low BM | $HML_t=\tfrac12(SH_t+BH_t)-\tfrac12(SL_t+BL_t)$ |
| RMW | Half small-robust OP, half big-robust OP | Half small-weak OP, half big-weak OP | $RMW_t=\tfrac12(SR_t+BR_t)-\tfrac12(SW_t+BW_t)$ |
| CMA | Half small-conservative investment, half big-conservative investment | Half small-aggressive investment, half big-aggressive investment | $CMA_t=\tfrac12(SC_t+BC_t)-\tfrac12(SA_t+BA_t)$ |
| FF5 SMB | Average of the three grids' small-cell portfolios | Average of the three grids' big-cell portfolios | $SMB_t=\tfrac13(SMB^{BM}_t+SMB^{OP}_t+SMB^{INV}_t)$ |

For each grid, $SMB^q$ is the average of its three small-cell returns minus the average of its three big-cell returns. FF5 averages these size spreads across BM, profitability and investment; conventional FF3 uses the BM spread alone. A positive characteristic-factor payoff means that its long leg outperformed its short leg.

### 2.3 Average returns and Sharpe ratios

The bar charts compare average excess returns and Sharpe ratios over the same months. Mean returns are annualized by multiplying monthly means by twelve; Sharpe ratios account for the volatility of those excess payoffs. The long–short factors already represent excess payoffs, so RF is not subtracted again.

In [ ]:
from factor_asset_pricing.construction import characteristic_panel, annual_ff5_assignments
from factor_asset_pricing.factors import fama_french_five_factors
from factor_asset_pricing.asset_pricing_tests import performance_summary
from factor_asset_pricing.french_data import load_french
from factor_asset_pricing.validation import validate_factor_returns

panel, annual = load_prepared_data(PROCESSED, RAW_WRDS)
measured = characteristic_panel(panel, annual)
assignments = annual_ff5_assignments(measured)
constructed = fama_french_five_factors(assignments['panel'],
    size_col='me_portfolio', value_col='bm_portfolio', profitability_col='op_portfolio',
    investment_col='inv_portfolio', return_col='retadj', risk_free_col='RF',
    weight_col='weight', id_col='permno', lag_weights=False)
prepared_factors = constructed['factors'].rename(columns={'date': 'mdate'})
factor_names = ['MKT', 'SMB', 'HML', 'RMW', 'CMA']
empirical_factors = prepared_factors.set_index('mdate')[factor_names].sort_index().dropna()
assert empirical_factors.index.is_unique and np.isfinite(empirical_factors.to_numpy()).all()
months = empirical_factors.index.to_period('M')
assert np.all(np.diff(months.astype('int64')) == 1)
print(f'Common sample: {months[0]}–{months[-1]} ({len(months)} months).')

stats = performance_summary(empirical_factors, hac_lags=12)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), layout='constrained')
colors = [NAVY, BLUE, GREEN, ORANGE, GRAY]
(100 * stats.mean_annualized).plot.bar(ax=axes[0], color=colors, rot=0)
stats.sharpe.plot.bar(ax=axes[1], color=colors, rot=0)
axes[0].set(title='Average excess returns', ylabel='Percent per year')
axes[1].set(title='Sharpe ratios', ylabel='Annualized Sharpe ratio')
for ax in axes:
    ax.axhline(0, color=GRAY, linewidth=.8)
    ax.set_xlabel('')
plt.show()

The market has the highest average excess return, while RMW has the highest Sharpe ratio. This shows why average returns and risk-adjusted performance need not give the same ranking. Neither measure establishes alpha relative to another benchmark.


### 2.4 Historical factor performance

Another way to compare performance is to follow cumulative payoffs over time. Each series starts at one and compounds its monthly excess payoffs. These are comparable factor-payoff indices rather than funded total-return wealth accounts, and the logarithmic scale shows proportional changes.

In [ ]:
assert empirical_factors.gt(-1).all().all()
index_paths = (1 + empirical_factors).cumprod()
initial_date = empirical_factors.index[0] - pd.offsets.MonthBegin(1)
initial = pd.DataFrame(1., index=[initial_date], columns=factor_names)
fig, ax = plt.subplots(figsize=(10, 4.7), layout='constrained')
pd.concat([initial, index_paths]).plot(ax=ax, logy=True, color=colors)
ax.set(title='Historical factor performance', xlabel='',
       ylabel='Cumulative excess-payoff index (log scale)')
ax.legend(ncol=5, frameon=False)
plt.show()

The market finishes well above the characteristic factors, with RMW showing the strongest cumulative path among them. The paths also show that gains are uneven over time. Their final ranking does not establish which factor adds most to a diversified portfolio.


### 2.5 Validation against published factors

Finally, I compare our factors with the published French series on matching dates. I use the time-series regressions explained in 05 to assess their agreement: correlations and loadings describe co-movement, while mean differences and regression alphas show remaining differences in average payoffs. Alpha statistics use Newey–West standard errors with twelve monthly lags.

In [ ]:
official = load_french('ff5', ROOT / 'data/raw/french')
validation_dates = empirical_factors.index.intersection(official.index)
official = official.loc[validation_dates]
assert official[factor_names].notna().all().all()
factor_validation = validate_factor_returns(empirical_factors.loc[validation_dates], official,
    factors=factor_names, hac_lags=12)
validation_table = factor_validation[['correlation', 'mean_difference', 'loading', 'alpha', 'alpha_t_statistic']].copy()
validation_table[['mean_difference', 'alpha']] *= 100
validation_table.columns = ['Correlation', 'Mean difference (% monthly)', 'Loading',
                            'Alpha (% monthly)', 'NW t(alpha)']
display(validation_table.round(3))

Correlations exceed 0.96 for all five factors, and the fitted loadings are close to one, indicating close agreement with the published series. Some average differences remain, with SMB and RMW retaining statistically significant regression alphas at the nominal 5% level. The construction therefore tracks the published factors closely without reproducing them exactly; differences in universe, accounting conventions and RF contribute to this distinction.


## References

- Fama, E. F., and French, K. R. (2015). A five-factor asset pricing model. *Journal of Financial Economics*, 116(1), 1–22. [DOI](https://doi.org/10.1016/j.jfineco.2014.10.010).
- Kenneth French, [five-factor construction documentation](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_5_factors_2x3.html).
